# 4. Mixture of Experts (MoE)

**Goal:** understand how a model can have many parameters but only use a few
per token.

```
token -> router -> top-k expert IDs -> weighted sum of expert outputs
      -> plus always-on shared experts
```

## Setup

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while not (REPO / "DeepSeekFlashV4-Mini").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "DeepSeekFlashV4-Mini"))
sys.path.insert(0, str(REPO))
import torch

torch.manual_seed(0)
print("repo root:", REPO.name)
print("torch:", torch.__version__)

repo root: deepseek_flash_lab
torch: 2.13.0+cu130


## 1. Build a tiny MoE

8 routed experts, 1 shared expert, 2 activated experts per token.

In [2]:
from model.moe import MoE

d_model, vocab = 16, 16
moe = MoE(
    d_model=d_model,
    n_routed=8,
    n_shared=1,
    n_activated=2,
    hidden=32,
    vocab_size=vocab,
)
total = sum(p.numel() for p in moe.parameters())
print("total parameters:", f"{total:,}")

total parameters: 13,952


In [3]:
x = torch.randn(4, 6, d_model)
log = []
y = moe(x, log=log)

print("input :", tuple(x.shape))
print("output:", tuple(y.shape))
print("routing log:", log)

input : (4, 6, 16)
output: (4, 6, 16)
routing log: ['MoE[learned] hist=[8, 4, 7, 1, 6, 5, 5, 12]']


## 2. Inspect the router

The router scores all experts, then keeps the top-k per token.

In [4]:
flat = x.reshape(-1, d_model)
topv, topi = moe.router(flat)
print("top-k values :", tuple(topv.shape), "-> [tokens, k]")
print("top-k indices:", tuple(topi.shape))

first = topi[:6]
print("first 6 tokens choose:", first.tolist())
print("their weights       :", topv[:6].round(decimals=3).tolist())

top-k values : (24, 2) -> [tokens, k]
top-k indices: (24, 2)
first 6 tokens choose: [[0, 1], [6, 7], [4, 7], [5, 4], [1, 6], [7, 1]]
their weights       : [[0.8330000042915344, 0.6669999957084656], [0.7519999742507935, 0.7480000257492065], [0.8119999766349792, 0.6880000233650208], [0.7929999828338623, 0.7070000171661377], [0.7540000081062317, 0.7459999918937683], [0.7839999794960022, 0.7160000205039978]]


In [5]:
import torch

hist = torch.bincount(topi.reshape(-1), minlength=8)
print("expert usage across 24 tokens:", hist.tolist())
print("min / max:", hist.min().item(), "/", hist.max().item())

expert usage across 24 tokens: [8, 4, 7, 1, 6, 5, 5, 12]
min / max: 1 / 12


**Notice:** usage is uneven. That is normal, but extreme imbalance is a known
MoE failure mode. The router has a small bias update that nudges it toward
balance during training.

## 3. Learned routing vs hash routing

V4-Flash uses a fixed hash for the first MoE layers and learned routing later.
Hash routing ignores the input content and uses only the token ID, which is
stable and cheap.

In [6]:
hash_moe = MoE(
    d_model=d_model, n_routed=8, n_shared=1, n_activated=2,
    hidden=32, vocab_size=vocab, use_hash=True,
)
tokens = torch.arange(12).reshape(2, 6)
_, hash_ids = hash_moe.router(tokens.reshape(-1), tokens=tokens)
print("hash routing for IDs 0..11:")
print(hash_ids[:, 0].tolist())
print("deterministic for the same ID:", hash_ids[:, 0].tolist() == hash_moe.router(tokens.reshape(-1), tokens=tokens)[1][:, 0].tolist())

hash routing for IDs 0..11:
[0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3]
deterministic for the same ID: True


## Recap

- Total parameters can be large while per-token compute stays small.
- Router chooses `k` experts per token.
- Shared experts are always active.
- Hash routing is fixed by token ID; learned routing depends on the vector.

**Try it:** `docs/EXERCISES.md` exercise 4 asks you to change `n_activated`
and predict the effect on compute and routing.